# 14 · Multi-hop Retrieval

**Idea:** some questions need a *chain* of facts — the answer to one lookup tells you what to search for next. Single-shot retrieval fails here. **Multi-hop** loops: retrieve, let an LLM propose the next query to fill the gap, retrieve again, then answer from everything gathered.

We use the `systems_and_teams.md` corpus, where the facts are deliberately spread across sections.

> Runs the loop on Ollama (several small calls).

In [1]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore

#corpus = load_markdown()
# Check if directory exists
corpus = load_markdown(directory=str(root / 'multihop'))
print('\n-----SNEHAL START---------------\n')
for co in corpus:
    print(f"Text: {co.text}\n")
    print(f"Source: {co.source}\n")
    print(f"Index: {co.index}\n")
    print(f"Meta: {co.meta}\n")
print('\n-----SNEHAL END---------------\n')
store = NumpyStore(provider='openai'); store.add(corpus)


# A question no single passage answers:
#   Helpdesk Assistant -> powered by Atlas -> Atlas calls Beacon -> built by Search Platform -> led by Priya Nair
#question = 'Who leads the team that built the reranker used by the Helpdesk Assistant?'
question = 'What year did the company that acquired the startup founded by Sam\'s college roommate go public?'


-----SNEHAL START---------------

Text: Silicon Valley veteran Alex Vance has maintained an incredibly prolific career over the last two decades, successfully launching five distinct technology companies.

Source: Alex_Vance_Entrepreneurship.md

Index: 0

Meta: {'heading': 'Alex Vance - Career', 'title': 'Founder Profile: Alex Vance Serial Entrepreneurship', 'tags': ['founder', 'venture-capital', 'startup-history', 'portfolio'], 'audience': 'engineers'}

Text: His entrepreneurial journey began with Vertex Data, an analytics tool that was eventually acquired by a larger enterprise. Following that exit, Vance founded Nexus Security, which grew steadily before also being acquired in a private transaction.

Source: Alex_Vance_Entrepreneurship.md

Index: 1

Meta: {'heading': 'Alex Vance - entrepreneurial journey', 'title': 'Founder Profile: Alex Vance Serial Entrepreneurship', 'tags': ['founder', 'venture-capital', 'startup-history', 'portfolio'], 'audience': 'engineers'}

Text: Seeking a 

First show that **single-hop** retrieval struggles: the top passages rarely contain the whole chain in one place.

In [3]:
for h in store.search(question, k=3):
    print(f'{h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

 0.47  Slaesforce_Listing.md :: Salesforce history
 0.42  Sam_College_Life.md :: Early Years
 0.39  Alex_Vance_Entrepreneurship.md :: Alex Vance - entrepreneurial journey


Now run multi-hop. Watch the `queries` trace — the LLM reformulates toward the missing fact at each hop.

In [6]:
from rag_lab import multihop_answer
from study_buddy import CostTracker

tracker = CostTracker()
answer, trace = multihop_answer(question, store, provider='openai', k=3, max_hops=3, tracker=tracker)

print('hops:')
for i, q in enumerate(trace.queries, 1):
    print(f'  {i}. {q}')
print('\nanswer:', answer)
print('\ncost:', tracker.summary())

----------------------------------------------------
You are researching to answer a multi-part QUESTION by chaining facts across documents.

Step 1: List the specific named facts the QUESTION requires (people, companies, dates, etc.).
Step 2: Check CONTEXT SO FAR against that list. Mark each fact FOUND only if its exact value is explicitly written in the context -- never mark something FOUND because you happen to know it from outside knowledge; that counts as MISSING.
Step 3: If anything is MISSING, decide the ONE search query most likely to retrieve it -- name the specific missing entity or relationship, don't just repeat the QUESTION verbatim. If every fact has an explicit, concrete value written in the context, you're done.

This is search 1 of at least 2 required searches for a question this complex -- you may NOT answer DONE yet, even if the context looks sufficient. Propose the next query.

QUESTION: What year did the company that acquired the startup founded by Sam's college ro

**Takeaway:** multi-hop turns retrieval into a loop steered by the model. It unlocks bridge questions that no single chunk answers — at the price of extra retrieval + reasoning calls, so cap `max_hops`.

Next → `15_agentic_rag.ipynb`